# practice # 8


SQL2TEXT

sql lite와 llm 을 활용하여 쿼리문 조회 및 평가 기능
[sql_lite 정보](https://www.sqlitetutorial.net/sqlite-sample-database/)

In [1]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from module.base_model import *

In [2]:
start_langsmith('practice_8')

LangSmith 추적을 시작합니다.
[프로젝트명]
practice_8


In [3]:
from typing import TypedDict, Annotated, List, Literal,Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks


1. **데이터베이스 스키마 파악**: 사용 가능한 테이블 목록을 가져옵니다.
2. **관련 테이블 선택**: 질문과 연관된 테이블을 선택합니다.
3. **DDL 조회**: 선택된 테이블의 스키마 정의(DDL)를 가져옵니다.
4. **쿼리 생성**: 질문과 DDL 정보에 기반하여 SQL 쿼리를 작성합니다.
5. **쿼리 점검**: LLM을 사용하여 일반적인 오류를 검토하고 쿼리를 개선합니다.
6. **쿼리 실행 및 오류 처리**: 데이터베이스 엔진에 쿼리를 실행하고, 오류 발생 시 수정하여 성공적으로 쿼리를 수행합니다.
7. **응답 생성**: 쿼리 결과를 기반으로 최종 답변을 제공합니다.

In [4]:
# DB 테이블 리스트 조회 => fist_tool_call
# 질문과 관련된 테이블 조회 => model_get_schema
# 쿼리 생성  => query_gen
# 쿼리 결과 분기 처리 
# 에러 아닌경우 쿼리  check => correct_query
# 쿼리 실행
# 최종 결과 LLM 답변

### 상태 생성

In [5]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

### 일반 함수 선언

In [6]:
# 오류 처리 함수
def handle_tool_error(state) -> dict:
    # 오류 정보 조회
    error = state.get("error")
    # 도구 정보 조회
    tool_calls = state["messages"][-1].tool_calls
    # ToolMessage 로 래핑 후 반환
    return {
        "messages": [
            ToolMessage(
                content=f"Here is the error: {repr(error)}\n\nPlease fix your mistakes.",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

def tool_node_with_fallback(tools:list) -> RunnableWithFallbacks[Any, dict]:
    """
    Create a ToolNode with a fallback to handle errors and surface them to the agent.
    """
    # 오류 발생 시 대체 동작을 정의하여 ToolNode에 추가
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )


# 쿼리 실행 부
@tool
def db_query_tool(query: str) -> str:
    """
    Run SQL queries against a database and return results
    Returns an error message if the query is incorrect
    If an error is returned, rewrite the query, check, and retry
    """
    # 쿼리 실행
    db = get_db()
    result = db.run_no_throw(query)

    # 오류: 결과가 없으면 오류 메시지 반환
    if not result:
        return "Error: Query failed. Please rewrite your query and try again."
    # 정상: 쿼리 실행 결과 반환
    return result

# 쿼리 오타 검정 
def get_query_check_node(query):
    prompt = get_prompt_query_check()
    llm = get_gpt().bind_tools([db_query_tool],tool_choice='db_query_tool')
    chain = prompt | llm 
    return  {"messages": [chain.invoke({"messages":query})]}

### 노드 선언

In [7]:

def get_table_list_node(state: State) -> dict[str, list[AIMessage]]:
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    llm_get_schema = llm.bind_tools([sql_db_list_tables],tool_choice='sql_db_list_tables')
    return State({ "messages": [llm_get_schema.invoke(state["messages"])]})

def get_all_table_node(state:State):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    return tool_node_with_fallback([sql_db_list_tables])

def get_one_table_info_node(state:State):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    llm_with_schema = llm.bind_tools([sql_db_schema],tool_choice='sql_db_schema')
    result = llm_with_schema.invoke(state['messages'])
    return State({'messages':[result]})

def get_one_table_schema_node(state:State):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    return tool_node_with_fallback([sql_db_schema])

def get_query_gen_node(state:State):
    prompt = get_prompt_query_gen()
    llm =get_gemini()
    query_gen_llm = prompt | llm.bind_tools([get_query_check_node])
    # query_gen_llm = prompt | llm
    history = state["messages"]
    query_gen =query_gen_llm.invoke({'placeholder':history})
    return State({ "messages": [query_gen]})



def execute_query(state:State):
    query = ''
    messages = state["messages"][-1]
    if len(messages.tool_calls) > 0:
        query = messages.tool_calls[0]['args']['query']
    else:
        query = state["messages"][-1].content
    return State({'messages':db_query_tool(query)})

def answer_node(state:State):
    prompt = get_prompt_query_gen()
    llm =get_gemini()
    query_gen_llm = prompt | llm
    history = state["messages"]
    answer =query_gen_llm.invoke({'placeholder':history})
    return State({ "messages": [answer]})

### 분기 수행

In [8]:
# routing 분기처리 수행 
def routing(state: State) -> Literal["get_query_gen_node", "answer_node"]:
    latest_messages:str = state["messages"][-1].content
    if "Error:" in latest_messages:
        return "get_query_gen_node"
    else:
        return "answer_node"

### 랭그래프 선언

In [9]:
state_graph = StateGraph(State)
state_graph.add_node('get_table_list_node',get_table_list_node)
state_graph.add_node('get_all_table_node',get_all_table_node)
state_graph.add_node('get_one_table_info_node',get_one_table_info_node)
state_graph.add_node('get_one_table_schema_node',get_one_table_schema_node)
state_graph.add_node("get_query_gen_node", get_query_gen_node)
state_graph.add_node("execute_query", execute_query)
state_graph.add_node("answer_node", answer_node)

# state_graph.add_node("get_query_check_node", get_query_check_node)



state_graph.add_edge(START,'get_table_list_node')
state_graph.add_edge('get_table_list_node','get_all_table_node')
state_graph.add_edge('get_all_table_node','get_one_table_info_node')
state_graph.add_edge('get_one_table_info_node','get_one_table_schema_node')
state_graph.add_edge('get_one_table_schema_node','get_query_gen_node')

state_graph.add_edge('get_query_gen_node','execute_query')
state_graph.add_conditional_edges(
    source='execute_query',
    path=routing
)

state_graph.add_edge('answer_node',END)

ck = get_check_pointer()
graph = state_graph.compile(checkpointer=ck)

### 시각화

In [10]:

# visualize_graph(graph)

SQL2TEXT 샘플 만들기용 질문 5가지를 작성해 드리겠습니다.

1. 가장 많이 판매된 상위 10개 상품의 이름과 총 판매 수량을 알려주세요.

2. 특정 고객이 지난 1년간 구매한 주문 내역과 총 결제 금액을 조회해주세요.

3. 각 카테고리별 상품의 평균 가격과 상품 수량 정보를 보여주세요.

4. 지난 달 발행된 청구서 번호와 해당 청구서에 포함된 항목별 상세 내역을 출력해주세요.

5. 특정 아티스트가 출시한 앨범 목록과 트랙 수, 총 재생 시간을 알려주세요.

In [11]:
uuid = get_random_uuid()
config = get_runnable_config(recursion_limit=10,thread_id=uuid)
# inputs  = {'messages':['Edwards	Nancy 직원의 인적정보를 모두 조회해줘']}
inputs  = {'messages':['How many albums does the artist Led Zeppelin have?']}
stream_graph(graph,inputs,config)



🔄 Node: get_table_list_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: get_all_table_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
🔄 Node: get_one_table_info_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: get_one_table_schema_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/
🔄 Node: get_query_gen_node

/tmp/ipykernel_365444/389454951.py:46: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return State({'messages':db_query_tool(query)})



🔄 Node: answer_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Answer: Led Zeppelin은 14개의 앨범을 가지고 있습니다.

In [12]:
snapshot = graph.get_state(config)
snapshot.values 

{'messages': [HumanMessage(content='How many albums does the artist Led Zeppelin have?', additional_kwargs={}, response_metadata={}, id='407b8b37-e005-4f36-82ef-9e405a0606f4'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'sql_db_list_tables', 'arguments': '{"tool_input": ""}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--f07a6b09-0567-47ab-8496-527ed06376c9', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'c8e54386-6763-4ccb-b2c8-9601140334c0', 'type': 'tool_call'}], usage_metadata={'input_tokens': 76, 'output_tokens': 19, 'total_tokens': 95, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track', name='sql_db_list_tables', id='3afff085-1589-498e-9b8c-a3c182807379', tool_call_id='c8e54386-6763-4ccb-b2c8-9601140334c0'),
  AIMessage(content='', a

In [1]:


# db = SQLDatabase.from_uri("sqlite:///Chinook.db")

# # SQLDatabaseToolkit 생성
# toolkit = SQLDatabaseToolkit(db=db, llm=get_gpt())

# # SQLDatabaseToolkit에서 사용 가능한 도구 목록
# tools = toolkit.get_tools()

# #데이터베이스에서 사용 가능한 테이블을 나열하는 도구 선택
# list_tables_tool = next(tool for tool in tools if tool.name == "sql_db_list_tables")
# # 특정 테이블의 DDL을 가져오는 도구 선택
# get_schema_tool = next(tool for tool in tools if tool.name == "sql_db_schema")

# get_query_tool = next(tool for tool in tools if tool.name == "sql_db_query")

# get_check_tool = next(tool for tool in tools if tool.name == "sql_db_query_checker")

# query = 'SELECT T.Name, SUM(IL.Quantity) AS TotalQuantity FROM Track AS T JOIN InvoiceLine AS IL ON T.TrackId = IL.TrackId GROUP BY T.TrackId ORDER BY TotalQuantity DESC LIMIT 10;'

# print(get_query_tool.invoke(query))



NameError: name 'SQLDatabase' is not defined

In [ ]:
from langsmith import Client

# 클라이언트 초기화
client = Client()

# 데이터셋 생성 및 업로드
examples = [
    (
        "Which country's customers spent the most? And how much did they spend?",
        "The country whose customers spent the most is the USA, with a total spending of 523.06.",
    ),
    (
        "What was the most purchased track of 2013?",
        "The most purchased track of 2013 was Hot Girl.",
    ),
    (
        "How many albums does the artist Led Zeppelin have?",
        "Led Zeppelin has 14 albums",
    ),
    (
        "What is the total price for the album “Big Ones”?",
        "The total price for the album 'Big Ones' is 14.85",
    ),
    (
        "Which sales agent made the most in sales in 2009?",
        "Steve Johnson made the most sales in 2009",
    ),
]

dataset_name = "SQL Agent Response"

if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name)
    inputs, outputs = zip(
        *[({"input": text}, {"output": label}) for text, label in examples]
    )
    client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset.id)

In [ ]:
# 에이전트의 SQL 쿼리 응답을 예측하기 위한 함수 정의
def predict_sql_agent_answer(example: dict):
    """Use this for answer evaluation"""
    config = RunnableConfig(configurable={"thread_id": get_random_uuid()})

    inputs = {
        "messages": [HumanMessage(content=example["input"])],
    }
    # 그래프를 실행하여 메시지 결과 조회
    messages = graph.invoke(inputs, config)
    answer = messages["messages"][-1].content
    # 결과 반환
    return {"response": answer}

In [ ]:
from langchain import hub
from langchain_openai import ChatOpenAI

# Grade prompt
grade_prompt_answer_accuracy = hub.pull("langchain-ai/rag-answer-vs-reference")


# 답변 평가자 LLM-as-judge 정의
def answer_evaluator(run, example) -> dict:
    # input: 질문
    input_question = example.inputs["input"]
    # output: 참조 답변
    reference = example.outputs["output"]
    # 예측 답변
    prediction = run.outputs["response"]

    # LLM 평가자 초기화
    llm = get_gemini()
    answer_grader = grade_prompt_answer_accuracy | llm

    # 평가자 실행
    score = answer_grader.invoke(
        {
            "question": input_question,
            "correct_answer": reference,
            "student_answer": prediction,
        }
    )
    score = score["Score"]

    # 점수 반환
    return {"key": "answer_v_reference_score", "score": score}

In [ ]:
from langsmith.evaluation import evaluate

# 평가용 데이터셋 이름
dataset_name = "SQL Agent Response"

try:
    # 평가 진행
    experiment_results = evaluate(
        predict_sql_agent_answer,  # 평가시 활용할 예측 함수
        data=dataset_name,  # 평가용 데이터셋 이름
        evaluators=[answer_evaluator],  # 평가자 목록
        num_repetitions=3,  # 실험 반복 횟수 설정
        experiment_prefix="sql-agent-eval",
        metadata={"version": "chinook db, sql-agent-eval: gpt-4o"},  # 실험 메타데이터
    )
except Exception as e:
    print(e)